# test weather API hooks

This notebook testes the various PAI hooks for weather data and aims at studying correlation witzh ground weather data

In [1]:
import pandas as pd
import datetime as dt

In [2]:
from src.api_hooks.weather_sources import _fetch_era5

In [3]:
# data sources
local_weather_src =  "data/raw/land_management/weather_observations.csv"

In [4]:
# import and preprocess local_weather pattern

local_weather = pd.read_csv(local_weather_src)
local_weather["datetime"] = local_weather["timestamp"].apply(lambda x: dt.datetime.fromtimestamp(x))


def preprocess_locasl_weather(local_src: str)->pd.Dataframe:
    local_weather = pd.read_csv(local_weather_src)
    local_weather["datetime"] = local_weather["timestamp"].apply(lambda x: dt.datetime.fromtimestamp(x))
    return local_weather

In [5]:
st_date= local_weather["datetime"].sort_values().iloc[0]
en_date= local_weather["datetime"].sort_values().iloc[-1]
st_date, en_date

(Timestamp('2023-02-19 02:00:00'), Timestamp('2026-03-26 19:14:00'))

## Testing weather sources
4 weather data sources were collected for comparison with the local weather station data: [era5](https://earthdatahub.destine.eu/collections/era5), [imerg](https://gpm.nasa.gov/data/imerg), [prism](https://prism.oregonstate.edu/) and [texmesonet](https://txwaterdatahub.org)


API hooks were developed for all of them.

Here we will compare them with the loca data to see which one is closer to local data

In [6]:
from src.api_hooks import get_precipitation

In [7]:
location_point= {"type": "Point", "coordinates": [-105.0885872, 30.8128463]}
st_date_str = dt.datetime.strftime(st_date, '%Y-%m-%d')
en_date_str = "2023-12-31"

In [8]:
import toml
SETTINGS = toml.load("/home/matt/code/WASP_dashboard/src/api_hooks/settings.toml")

Redoing all api hooks, beacuse Claude was not able to give me somthing that works

In [9]:
from src.api_hooks._utils import Context

CONTEXT = Context("src/api_hooks/settings.toml")
CONTEXT

time_frame: ['2025-02-01', '2025-02-28'], location: POINT (-105.0885872 30.8128463), folders: [temp] data/raw/weather/temp, [processed] data/raw/weather/processed

## ERA5 donwload

In [49]:
from src.api_hooks._era5 import fetch_era5, process_era5

In [ ]:
# raw=fetch_era5(CONTEXT)
# daily=process_era5(raw, CONTEXT)

# IMERG

In [ ]:
import earthaccess
from typing import Any, List
from src.api_hooks._utils import Context
from concurrent.futures import ThreadPoolExecutor
from functools import partial
import xarray as xr
import pathlib as p

IMERG_SETTINGS = dict(
    strategy="environment",
    short_name="GPM_3IMERGDF",
    version="07",
    variables="total_precipitation",
)


def _process_granule(
    path: str,
    west: float,
    south: float,
    east: float,
    north: float,
) -> xr.DataArray:
    """Open one IMERG v07 granule (local file), clip to bbox."""
    raw = xr.open_dataset(path, engine="h5netcdf")
    precip = raw["precipitation"]

    # Build slices that respect coordinate direction (IMERG lat/lon
    # can be ascending or descending; a wrong-direction slice silently
    # returns an empty array rather than erroring).
    lat = precip["lat"]
    lon = precip["lon"]
    lat_slice = slice(south, north) if float(lat[0]) <= float(lat[-1]) else slice(north, south)
    lon_slice = slice(west, east) if float(lon[0]) <= float(lon[-1]) else slice(east, west)

    sub = precip.sel(lat=lat_slice, lon=lon_slice).load()
    return sub.transpose(..., "lat", "lon")


def identify_imerg(context: Context, settings: dict = IMERG_SETTINGS) -> list:
    """Search EarthAccess for IMERG granules covering context's time/location."""
    earthaccess.login(strategy=settings["strategy"])
    w, s, e, n = context.location_buffer.bounds
    return earthaccess.search_data(
        short_name=settings["short_name"],
        version=settings["version"],
        temporal=(context.time_frame[0], context.time_frame[1]),
        bounding_box=(w, s, e, n),
    )


def download_imerg(
    imerg_granules: List[Any],
    context: Context,
    settings: dict = IMERG_SETTINGS,
    save_raw: bool = True,
):
    """Download and spatially clip IMERG granules to context's location buffer."""
    # Recompute bbox from context — w/s/e/n are not inherited from identify_imerg.
    w, s, e, n = context.location_buffer.bounds
    process = partial(_process_granule, west=w, south=s, east=e, north=n)

    paths = earthaccess.download(imerg_granules, local_path=context.temp_folder)
    with ThreadPoolExecutor(max_workers=4) as pool:
        day_datasets = list(pool.map(process, paths))

    combined = xr.concat(day_datasets, dim="time")
    if save_raw:
        out_path = (
            p.Path(context.temp_folder)
            / f"imerg_raw_{context.time_frame[0]}-{context.time_frame[1]}.nc"
        )
        combined.to_netcdf(out_path)
        return out_path
    return combined.to_dataset()


imerg_granules = identify_imerg(CONTEXT)
print(imerg_granules)
combined = download_imerg(imerg_granules, CONTEXT)
print(combined)


In [23]:
(CONTEXT.time_frame[0], CONTEXT.time_frame[1])

('2026-02-01', '2026-02-28')

In [ ]:
import tempfile

from pathlib import Path

import xarray as xr
from dotenv import load_dotenv

from src.api_hooks._utils import bbox

def process_imerg(raw_imerg: str | xr.Dataset, CONTEXT, settings, save_raw):


ds = xr.Dataset({"precip_mm": combined})
ds["precip_mm"].attrs["units"] = "mm"
ds

/home/matt/miniconda3/envs/wasp/lib/python3.14/site-packages/earthaccess/store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/31 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/31 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/31 [00:00<?, ?it/s]

<xarray.Dataset> Size: 248B
Dimensions:    (lon: 0, lat: 0, time: 31)
Coordinates:
  * lon        (lon) float32 0B 
  * lat        (lat) float64 0B 
  * time       (time) datetime64[ns] 248B 2025-01-01 2025-01-02 ... 2025-01-31
Data variables:
    precip_mm  (time, lat, lon) float32 0B

## PRISM

In [ ]:
from datetime import datetime , timedelta
import zipfile
import io
from concurrent.futures import ThreadPoolExecutor
import tempfile
import numpy as np
import requests
from src.api_hooks._utils import bbox
from pathlib import Path
import pickle
_PRISM_URL = "https://services.nacse.org/prism/data/get/us/4km/ppt/{date}"


#-----------------------
def _date_range(start_time: str, end_time: str):
    """Yield (YYYYMMDD, YYYY-MM-DD) pairs from start_time to end_time inclusive."""
    current = datetime.strptime(start_time, "%Y-%m-%d")
    end = datetime.strptime(end_time, "%Y-%m-%d")
    while current <= end:
        yield current.strftime("%Y%m%d"), current.strftime("%Y-%m-%d")
        current += timedelta(days=1)


def _fetch_day(
    date_yyyymmdd: str,
    west: float,
    south: float,
    east: float,
    north: float,
) -> xr.DataArray:
    """Download and parse one day's PRISM precipitation GeoTIFF raster."""
    import rasterio
    from rasterio.windows import from_bounds

    url = _PRISM_URL.format(date=date_yyyymmdd)
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()

    if not resp.content[:4] == b"PK\x03\x04":
        raise RuntimeError(
            f"PRISM returned a non-ZIP response for {date_yyyymmdd} "
            f"(likely rate-limited): {resp.text[:300]}"
        )

    with tempfile.TemporaryDirectory() as tmpdir:
        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            zf.extractall(tmpdir)

        tif_files = list(Path(tmpdir).glob("*.tif"))
        if not tif_files:
            raise FileNotFoundError(f"No .tif file in PRISM ZIP for {date_yyyymmdd}")

        with rasterio.open(tif_files[0]) as src:
            window = from_bounds(west, south, east, north, src.transform)
            data = src.read(1, window=window, masked=True).filled(np.nan).astype(float)
            win_transform = src.window_transform(window)

    height, width = data.shape
    lons = win_transform.c + (np.arange(width) + 0.5) * win_transform.a
    lats = win_transform.f + (np.arange(height) + 0.5) * win_transform.e

    return xr.DataArray(data, dims=["lat", "lon"], coords={"lat": lats, "lon": lons})


def _fetch_prism(start_time: str, end_time: str, location: dict, save_pickle: bool=True) -> xr.Dataset:
    """
    Fetch PRISM daily precipitation for a location and date range.

    Parameters
    ----------
    start_time : str  ISO date, e.g. "2024-01-01"
    end_time   : str  ISO date, e.g. "2024-01-03"
    location   : dict GeoJSON Point or Polygon

    Returns
    -------
    xr.Dataset with variable precip_mm (time, lat, lon), units mm/day
    """
    w, s, e, n = bbox(location, buffer=0.1)
    date_pairs = list(_date_range(start_time, end_time))

    def fetch_day(pair):
        date_compact, date_iso = pair
        da = _fetch_day(date_compact, w, s, e, n)
        return da.expand_dims({"time": [np.datetime64(date_iso)]})

    with ThreadPoolExecutor(max_workers=4) as pool:
        arrays = list(pool.map(fetch_day, date_pairs))
    if save_pickle:
        pickle.dump(arrays)
        
    return arrays

def _process_prism(arrays, local_folder: str | p.Path):
    combined = xr.concat(arrays, dim="time")
    ds = xr.Dataset({"precip_mm": combined})
    ds["precip_mm"].attrs["units"] = "mm/day"
    return ds

In [ ]:
st_date_str= "2026-01-01"
en_date_str="2026-01-10"
_fetch_prism(st_date_str, en_date_str, location_point)

<xarray.Dataset> Size: 2kB
Dimensions:    (time: 10, lat: 5, lon: 5)
Coordinates:
  * time       (time) datetime64[s] 80B 2026-01-01 2026-01-02 ... 2026-01-10
  * lat        (lat) float64 40B 30.89 30.85 30.81 30.77 30.73
  * lon        (lon) float64 40B -105.2 -105.1 -105.1 -105.0 -105.0
Data variables:
    precip_mm  (time, lat, lon) float64 2kB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0